# 2. Open RadDB object and filter

Tutorial 1 wrote an archive. This one reads it back and filters it down.

**`RadDB` is one class with two roles.**

| role | what it is |
|---|---|
| *archive-bound* | knows where an archive lives, and reads from it |
| *data-carrying*  | holds the gates you loaded, and narrows them down |

`open()` turns the first into the second. Every operation on a data-carrying
RadDB returns a **new** one, so calls chain and nothing is changed in place.

---

In [1]:
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import polars as pl

import raddb

In [2]:
# --------------------------------------------------------------------------
# CONFIGURATION — edit these two paths to point at your own data
# --------------------------------------------------------------------------
# ARCHIVE_DIR must be the same archive tutorial 1 wrote.

FMI_DIR = Path("~/Desktop/LTE_project/ltenas8/data/RADAR/FMI_datatree_zarr").expanduser()
ARCHIVE_DIR = Path("~/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive").expanduser()

print("FMI DataTrees:", FMI_DIR)
print("Archive      :", ARCHIVE_DIR)

FMI DataTrees: /home/erik_poschivo/Desktop/LTE_project/ltenas8/data/RADAR/FMI_datatree_zarr
Archive      : /home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive


In [3]:
# This notebook stands on its own: build the archive if tutorial 1 has not run.
if not (ARCHIVE_DIR / "FANJ" / "LUT").exists():
    print("building the archive (see tutorial 1) ...")
    raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=3067).archive(
        datatree_dir=FMI_DIR,
        time_period=("2024-06-10", "2024-06-18"),
    )
else:
    print("archive already present:", ARCHIVE_DIR)

archive already present: /home/erik_poschivo/Desktop/LTE_project/ltenas8/users/giacobbi/raddb_tutorial_archive


## 1. `open()`: reading the archive

Reading never needs a CRS: it is recovered from the archive itself.

In [4]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR)
rdf = db.open(radars="FANJ")
rdf.head()

gate_id,time,CSP,DBZH,DBZHC,DBZV,HCLASS,KDP,LOG,PHIDP,PMI,RHOHV,SNR,SQIH,TH,TV,VRADDH,VRADH,WRADH,ZDR,ZDRC,volume_time,radar
i64,datetime[ns],f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,"datetime[μs, UTC]",str
713647000001015250,2024-06-10 12:00:07.469453312,16.35,0.39,0.39,17.65,1.0,0.0,17.48,11.546984,0.313018,0.794027,21.639999,0.939008,16.76,-327.679993,2.07,1.07,0.8,-4.16,-4.16,2024-06-10 12:00:02 UTC,"""FANJ"""
713647000001029750,2024-06-10 12:00:07.469453312,12.23,2.28,2.28,10.83,1.0,0.0,14.63,69.397263,0.454931,0.976912,17.48,0.978972,14.59,-327.679993,-1.15,-2.18,0.28,0.31,0.31,2024-06-10 12:00:02 UTC,"""FANJ"""
713647000001127750,2024-06-10 12:00:07.469453312,-0.02,0.24,0.24,0.47,1.0,0.0,3.65,67.249367,0.441457,0.910213,1.19,0.39722,0.2,-327.679993,-4.6,-5.6,1.46,-0.03,-0.03,2024-06-10 12:00:02 UTC,"""FANJ"""
713647000001128250,2024-06-10 12:00:07.469453312,-0.01,5.0,5.0,4.93,4.0,0.0,6.91,73.92926,0.534052,0.986831,5.92,0.574947,4.99,-327.679993,-4.38,-5.44,1.51,-0.48,-0.48,2024-06-10 12:00:02 UTC,"""FANJ"""
713647000001128750,2024-06-10 12:00:07.469453312,0.02,4.81,4.81,4.56,1.0,0.0,6.72,75.461899,0.438314,0.949003,5.68,0.659805,4.83,-327.679993,-4.54,-5.6,1.16,-0.92,-0.92,2024-06-10 12:00:02 UTC,"""FANJ"""


`open()` narrows *before* anything is loaded — the time range, the radars and the
columns are all pushed down into the Parquet scan, so you never pay for data you
did not ask for.

In [5]:
# Only two variables, only radar FANJ
small_df = db.open(radars="FANJ", columns=["DBZH", "ZDR"])
print(f"small_df:\tcolumns: {small_df.columns()}\nsmall_df:\tgates: {len(small_df):,}")
print("-------------------------------")
# time period — the convective afternoon of 17 June 2024
day_df = db.open(radars="FANJ", time_period=("2024-06-17 12:00", "2024-06-17 18:00"))
print(f"day_df:\t\tcolumns: {day_df.columns()}\nday_df:\t\tgates: {len(day_df):,}")

small_df:	columns: ['gate_id', 'DBZH', 'ZDR', 'volume_time', 'radar']
small_df:	gates: 14,271,870
-------------------------------


day_df:		columns: ['gate_id', 'time', 'CSP', 'DBZH', 'DBZHC', 'DBZV', 'HCLASS', 'KDP', 'LOG', 'PHIDP', 'PMI', 'RHOHV', 'SNR', 'SQIH', 'TH', 'TV', 'VRADDH', 'VRADH', 'WRADH', 'ZDR', 'ZDRC', 'volume_time', 'radar']
day_df:		gates: 13,647,324


In [6]:
# Filters can be pushed down at open() too, so filtered-out rows are never materialised
filtered_df = db.open(radars="FANJ", filters={"var": "DBZH", "logic": ">", "threshold": 30})
print(f"before:\t{len(rdf):,} gates\t(with DBZH > 0 dBz)\nafter:\t{len(filtered_df):,}  gates\t(with DBZH > 30 dBz)")

before:	14,271,870 gates	(with DBZH > 0 dBz)
after:	781,623  gates	(with DBZH > 30 dBz)


## 2. What you are holding

The data lives in `.data` as a **polars** DataFrame. Polars is the backend
throughout RadDB (the read path, the LUT, the archive writer).

In [7]:
print("type:\t\t", type(rdf.data))
print("name type:\t", type(rdf.data).__name__)
print("shape:\t\t", rdf.data.shape)
rdf.data.head()

type:		 <class 'polars.dataframe.frame.DataFrame'>
name type:	 DataFrame
shape:		 (14271870, 23)


gate_id,time,CSP,DBZH,DBZHC,DBZV,HCLASS,KDP,LOG,PHIDP,PMI,RHOHV,SNR,SQIH,TH,TV,VRADDH,VRADH,WRADH,ZDR,ZDRC,volume_time,radar
i64,datetime[ns],f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,"datetime[μs, UTC]",str
713647000001015250,2024-06-10 12:00:07.469453312,16.35,0.39,0.39,17.65,1.0,0.0,17.48,11.546984,0.313018,0.794027,21.639999,0.939008,16.76,-327.679993,2.07,1.07,0.8,-4.16,-4.16,2024-06-10 12:00:02 UTC,"""FANJ"""
713647000001029750,2024-06-10 12:00:07.469453312,12.23,2.28,2.28,10.83,1.0,0.0,14.63,69.397263,0.454931,0.976912,17.48,0.978972,14.59,-327.679993,-1.15,-2.18,0.28,0.31,0.31,2024-06-10 12:00:02 UTC,"""FANJ"""
713647000001127750,2024-06-10 12:00:07.469453312,-0.02,0.24,0.24,0.47,1.0,0.0,3.65,67.249367,0.441457,0.910213,1.19,0.39722,0.2,-327.679993,-4.6,-5.6,1.46,-0.03,-0.03,2024-06-10 12:00:02 UTC,"""FANJ"""
713647000001128250,2024-06-10 12:00:07.469453312,-0.01,5.0,5.0,4.93,4.0,0.0,6.91,73.92926,0.534052,0.986831,5.92,0.574947,4.99,-327.679993,-4.38,-5.44,1.51,-0.48,-0.48,2024-06-10 12:00:02 UTC,"""FANJ"""
713647000001128750,2024-06-10 12:00:07.469453312,0.02,4.81,4.81,4.56,1.0,0.0,6.72,75.461899,0.438314,0.949003,5.68,0.659805,4.83,-327.679993,-4.54,-5.6,1.16,-0.92,-0.92,2024-06-10 12:00:02 UTC,"""FANJ"""


In [8]:
print("radars    :", rdf.radars())
print("variables :", rdf.columns())
print("time range:", rdf.start_time(), "->", rdf.end_time())
print("lon/lat    :", [round(v, 3) for v in rdf.geographic_extent()])
print("archive CRS:", rdf.crs())  # recovered from the archive itself

radars    : ['FANJ']
variables : ['gate_id', 'time', 'CSP', 'DBZH', 'DBZHC', 'DBZV', 'HCLASS', 'KDP', 'LOG', 'PHIDP', 'PMI', 'RHOHV', 'SNR', 'SQIH', 'TH', 'TV', 'VRADDH', 'VRADH', 'WRADH', 'ZDR', 'ZDRC', 'volume_time', 'radar']
time range: 2024-06-10 12:00:02+00:00 -> 2024-06-17 17:45:04+00:00


lon/lat    : [22.491, 31.725, 58.659, 63.134]
archive CRS: EPSG:3067


## 3. `filter()`: threshold on values

A filter is a plain dict: `{"var", "logic", "threshold"}`

In [9]:
rain = rdf.filter({"var": "DBZH", "logic": ">", "threshold": 20})
print(f"DBZH > 20: {len(rain):,} gates")

filt_df = rdf.filter(
    [
        {"var": "DBZH", "logic": ">", "threshold": 20},
        {"var": "RHOHV", "logic": ">=", "threshold": 0.98},
        {"var": "ZDR", "logic": ">", "threshold": 4},
    ],
)
print(f"DBZH > 20, RHOHV >= 0.98, ZDR > 4 : {len(filt_df):,} gates")

DBZH > 20: 2,603,259 gates
DBZH > 20, RHOHV >= 0.98, ZDR > 4 : 1,041 gates


## 4. `sel()`: select by label, xarray-style

Where `filter()` thresholds *values*, `sel()` selects by **coordinate**: a time, a
sweep, a range window, a longitude/latitude box. Scalars match exactly, `slice`
gives a closed interval, and a list matches any of its members.

In [10]:
print("one sweep      :", f"{len(rdf.sel(sweep=1)):,}")
print("sweeps 1,2,3   :", f"{len(rdf.sel(sweep=[1, 2, 3])):,}")
print("range 10-50 km :", f"{len(rdf.sel(range=slice(10_000, 50_000))):,}")
print("a lon/lat box  :", f"{len(rdf.sel(lon=slice(26.6, 27.6), lat=slice(60.6, 61.2))):,}")

one sweep      : 1,906,295


sweeps 1,2,3   : 4,425,869


range 10-50 km : 4,869,671


a lon/lat box  : 4,516,893


`range`, `azimuth`, `elevation_angle`, `latitude`, `longitude`
and `altitude` are **not stored in the Parquet files** — they live once in the LUT.
`sel()` borrows the column it needs, evaluates the selection, and drops it again,
so selecting on geometry costs no storage.

In [11]:
print("stored per gate:", rdf.columns())
print("also selectable :", ["range", "azimuth", "elevation_angle", "latitude", "longitude", "altitude", "sweep"])

narrow = rdf.sel(sweep=1, range=slice(20_000, 60_000))
print(f"\nsweep 1, 20-60 km: {len(narrow):,} gates " f"(columns unchanged: {narrow.columns() == rdf.columns()})")

stored per gate: ['gate_id', 'time', 'CSP', 'DBZH', 'DBZHC', 'DBZV', 'HCLASS', 'KDP', 'LOG', 'PHIDP', 'PMI', 'RHOHV', 'SNR', 'SQIH', 'TH', 'TV', 'VRADDH', 'VRADH', 'WRADH', 'ZDR', 'ZDRC', 'volume_time', 'radar']
also selectable : ['range', 'azimuth', 'elevation_angle', 'latitude', 'longitude', 'altitude', 'sweep']



sweep 1, 20-60 km: 412,692 gates (columns unchanged: True)


## 5. `add_feature()`: compute columns

`add_feature()` adds a column derived from the ones you already have and returns a
new RadDB, so it drops straight into a pipeline. The function receives the polars
frame; return a Series, a numpy array, or a polars expression.

In [12]:
derived = rdf.add_feature("DBZH_lin", lambda df: 10 ** (df["DBZH"] / 10)).add_feature(
    "DBZH_dev",
    lambda df: df["DBZH"] - df["DBZH"].mean(),
)
derived.head()

gate_id,time,CSP,DBZH,DBZHC,DBZV,HCLASS,KDP,LOG,PHIDP,PMI,RHOHV,SNR,SQIH,TH,TV,VRADDH,VRADH,WRADH,ZDR,ZDRC,volume_time,radar,DBZH_lin,DBZH_dev
i64,datetime[ns],f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,"datetime[μs, UTC]",str,f32,f32
713647000001015250,2024-06-10 12:00:07.469453312,16.35,0.39,0.39,17.65,1.0,0.0,17.48,11.546984,0.313018,0.794027,21.639999,0.939008,16.76,-327.679993,2.07,1.07,0.8,-4.16,-4.16,2024-06-10 12:00:02 UTC,"""FANJ""",1.093956,-11.687351
713647000001029750,2024-06-10 12:00:07.469453312,12.23,2.28,2.28,10.83,1.0,0.0,14.63,69.397263,0.454931,0.976912,17.48,0.978972,14.59,-327.679993,-1.15,-2.18,0.28,0.31,0.31,2024-06-10 12:00:02 UTC,"""FANJ""",1.690441,-9.797352
713647000001127750,2024-06-10 12:00:07.469453312,-0.02,0.24,0.24,0.47,1.0,0.0,3.65,67.249367,0.441457,0.910213,1.19,0.39722,0.2,-327.679993,-4.6,-5.6,1.46,-0.03,-0.03,2024-06-10 12:00:02 UTC,"""FANJ""",1.056818,-11.837352
713647000001128250,2024-06-10 12:00:07.469453312,-0.01,5.0,5.0,4.93,4.0,0.0,6.91,73.92926,0.534052,0.986831,5.92,0.574947,4.99,-327.679993,-4.38,-5.44,1.51,-0.48,-0.48,2024-06-10 12:00:02 UTC,"""FANJ""",3.162278,-7.077352
713647000001128750,2024-06-10 12:00:07.469453312,0.02,4.81,4.81,4.56,1.0,0.0,6.72,75.461899,0.438314,0.949003,5.68,0.659805,4.83,-327.679993,-4.54,-5.6,1.16,-0.92,-0.92,2024-06-10 12:00:02 UTC,"""FANJ""",3.026913,-7.267352


If you would rather work in plain polars or pandas, nothing stops you — `.data`
is an ordinary polars frame, and `to_pandas()` gives an ordinary pandas one.

In [13]:
rdf.data.with_columns((pl.col("DBZH") - pl.col("ZDR")).alias("DIFF"))

df = rdf.to_pandas()
df["DIFF"] = df["DBZH"] - df["ZDR"]

print(f"rdf type: {type(rdf.data)}")
print(f"df  type: {type(df)}")
df.head()

rdf type: <class 'polars.dataframe.frame.DataFrame'>
df  type: <class 'pandas.DataFrame'>


,gate_id,time,CSP,DBZH,DBZHC,DBZV,HCLASS,KDP,LOG,PHIDP,...,TH,TV,VRADDH,VRADH,WRADH,ZDR,ZDRC,volume_time,radar,DIFF
0,713647000001015250,2024-06-10 12:00:07.469453312,16.35,0.39,0.39,17.65,1.0,0.0,17.48,11.546984,...,16.76,-327.679993,2.07,1.07,0.80,-4.16,-4.16,2024-06-10 12:00:02+00:00,FANJ,4.55
1,713647000001029750,2024-06-10 12:00:07.469453312,12.23,2.28,2.28,10.83,1.0,0.0,14.63,69.397263,...,14.59,-327.679993,-1.15,-2.18,0.28,0.31,0.31,2024-06-10 12:00:02+00:00,FANJ,1.97
2,713647000001127750,2024-06-10 12:00:07.469453312,-0.02,0.24,0.24,0.47,1.0,0.0,3.65,67.249367,...,0.20,-327.679993,-4.60,-5.60,1.46,-0.03,-0.03,2024-06-10 12:00:02+00:00,FANJ,0.27
3,713647000001128250,2024-06-10 12:00:07.469453312,-0.01,5.00,5.00,4.93,4.0,0.0,6.91,73.929260,...,4.99,-327.679993,-4.38,-5.44,1.51,-0.48,-0.48,2024-06-10 12:00:02+00:00,FANJ,5.48
4,713647000001128750,2024-06-10 12:00:07.469453312,0.02,4.81,4.81,4.56,1.0,0.0,6.72,75.461899,...,4.83,-327.679993,-4.54,-5.60,1.16,-0.92,-0.92,2024-06-10 12:00:02+00:00,FANJ,5.73


## 6. Framework converter

The gates can leave RadDB as a pandas DataFrame, a geopandas GeoDataFrame, or an
xarray DataTree — three converters for three different frameworks.

### `to_pandas()`: the DataFrame

`to_pandas()` returns the loaded gates as an ordinary pandas DataFrame. On its own
it hands back exactly what is stored per gate: `gate_id`, `time`, the polarimetric variables, and
the `volume_time` / `radar` labels.

Geometry is **not** stored per gate — it lives once in the LUT — so it is merged
on `gate_id` only when you ask for it:

| call | columns added |
|---|---|
| `to_pandas()` | nothing; the stored columns only (dynamic variables) |
| `to_pandas(with_geometry=True)` | `latitude`, `longitude`, `altitude`, `sweep` |
| `to_pandas(with_polar_coords=True)` | the same, **plus** `range`, `azimuth`, `elevation_angle` |

`with_polar_coords` implies `with_geometry`. The polar coordinates are off by
default because they repeat what the Cartesian columns already say, unless you are
working in polar space.

Note what is **not** included: `x`, `y`, `z` — metres from the radar — are never
added by either flag, and the projected `x_<epsg>` / `y_<epsg>` appear only under a
condition. The next three cells explain why, and how to load all of them.

In [14]:
# No flags: the stored columns only, exactly as open() loaded them.
df = rain.to_pandas()
print("to_pandas():", type(df).__name__, df.shape)
print(list(df.columns))

to_pandas(): DataFrame (2603259, 23)
['gate_id', 'time', 'CSP', 'DBZH', 'DBZHC', 'DBZV', 'HCLASS', 'KDP', 'LOG', 'PHIDP', 'PMI', 'RHOHV', 'SNR', 'SQIH', 'TH', 'TV', 'VRADDH', 'VRADH', 'WRADH', 'ZDR', 'ZDRC', 'volume_time', 'radar']


In [15]:
# with_geometry=True joins the per-gate coordinates from the LUT on gate_id.
df_geo = rain.to_pandas(with_geometry=True)
print("added by with_geometry     :", [c for c in df_geo.columns if c not in df.columns])

added by with_geometry     : ['latitude', 'longitude', 'altitude', 'sweep']


In [16]:
# with_polar_coords=True also brings the polar coordinates the geometry came from.
df_polar = rain.to_pandas(with_polar_coords=True)
print("added by with_polar_coords :", [c for c in df_polar.columns if c not in df.columns])
df_polar.head(3)

added by with_polar_coords : ['latitude', 'longitude', 'altitude', 'sweep', 'range', 'azimuth', 'elevation_angle']


,gate_id,time,CSP,DBZH,DBZHC,DBZV,HCLASS,KDP,LOG,PHIDP,...,ZDRC,volume_time,radar,latitude,longitude,altitude,sweep,range,azimuth,elevation_angle
0,713647000001133250,2024-06-10 12:00:07.469453312,0.07,21.430000,21.430000,21.850000,4.0,0.13,21.959999,74.028137,...,0.42,2024-06-10 12:00:02+00:00,FANJ,62.102001,27.112360,1881.614041,0,133250.0,0.1,0.3
1,713647000001133750,2024-06-10 12:00:07.469453312,0.07,24.160000,24.160000,24.620001,4.0,0.12,24.639999,74.379715,...,0.58,2024-06-10 12:00:02+00:00,FANJ,62.106496,27.112377,1892.087740,0,133750.0,0.1,0.3
2,713647000001134250,2024-06-10 12:00:07.469453312,0.92,28.049999,28.049999,29.440001,3.0,0.12,26.650000,74.401688,...,1.00,2024-06-10 12:00:02+00:00,FANJ,62.110991,27.112393,1902.590849,0,134250.0,0.1,0.3


### Where the geometry lives

Two things are easy to trip over:

- **`x_<epsg>` / `y_<epsg>` appear only if the RadDB was created with `crs=`.**
  `crs()` reports the archive's projection either way, but the converters add the
  projected pair only when a projection was asked for explicitly.
- **`x` / `y` / `z`** — metres from the radar — are LUT columns that no converter
  attaches. Join the LUT yourself to get them, or any other LUT column.

In [17]:
# Projected coordinates: state the CRS when creating the RadDB, and
# with_geometry=True then adds x_<epsg> / y_<epsg> alongside lon/lat/alt.
db_proj = raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=3067)
rain_proj = db_proj.open(radars="FANJ", filters={"var": "DBZH", "logic": ">", "threshold": 20})

print("without crs= :", list(rain.to_pandas(with_geometry=True).columns))
print("with crs=3067:", list(rain_proj.to_pandas(with_geometry=True).columns))

without crs= : ['gate_id', 'time', 'CSP', 'DBZH', 'DBZHC', 'DBZV', 'HCLASS', 'KDP', 'LOG', 'PHIDP', 'PMI', 'RHOHV', 'SNR', 'SQIH', 'TH', 'TV', 'VRADDH', 'VRADH', 'WRADH', 'ZDR', 'ZDRC', 'volume_time', 'radar', 'latitude', 'longitude', 'altitude', 'sweep']


with crs=3067: ['gate_id', 'time', 'CSP', 'DBZH', 'DBZHC', 'DBZV', 'HCLASS', 'KDP', 'LOG', 'PHIDP', 'PMI', 'RHOHV', 'SNR', 'SQIH', 'TH', 'TV', 'VRADDH', 'VRADH', 'WRADH', 'ZDR', 'ZDRC', 'volume_time', 'radar', 'latitude', 'longitude', 'altitude', 'sweep', 'x_3067', 'y_3067']


In [18]:
# Any LUT column can be attached by joining on gate_id.  This is also how you add
# geometry to a frame loaded with open(), and the only way to get x / y / z
# (metres from the radar), which no converter attaches.
geometry = db.get_lut("FANJ").select(["gate_id", "x", "y", "z", "x_3067", "y_3067"])
joined = rain.data.join(geometry, on="gate_id", how="left")
joined.select(["gate_id", "DBZH", "x", "y", "z", "x_3067", "y_3067"]).head()

gate_id,DBZH,x,y,z,x_3067,y_3067
i64,f32,f64,f64,f64,f64,f64
713647000001133250,21.43,232.523651,133226.102929,1742.614041,505865.444342,6.8855e6
713647000001133750,24.16,233.395944,133725.889621,1753.08774,505865.418512,6.8860e6
713647000001134250,28.049999,234.268235,134225.67508,1763.590849,505865.392396,6.8865e6
713647000001134750,30.290001,235.140524,134725.459302,1774.12337,505865.365994,6.8871e6
713647000001135250,32.630001,236.012811,135225.242283,1784.685302,505865.339306,6.8876e6


### `to_geopandas()` — points with a CRS

In [19]:
# geopandas: point geometry per gate, ready for spatial joins or QGIS
gdf = rain.to_geopandas()
print("to_geopandas: ", type(gdf))
print("CRS:", gdf.crs)
gdf[["gate_id", "DBZH", "geometry"]].head()

to_geopandas:  <class 'geopandas.geodataframe.GeoDataFrame'>
CRS: EPSG:4326


,gate_id,DBZH,geometry
0,713647000001133250,21.430000,POINT (27.11236 62.102)
1,713647000001133750,24.160000,POINT (27.11238 62.1065)
2,713647000001134250,28.049999,POINT (27.11239 62.11099)
3,713647000001134750,30.290001,POINT (27.11241 62.11549)
4,713647000001135250,32.630001,POINT (27.11242 62.11998)


### `to_datatree()` — back to xarray

In [20]:
# DataTree: the full polar structure, for xarray workflows.
# A DataTree describes ONE volume: each sweep is an (azimuth x range) grid and
# time is a per-ray coordinate, so there is no dimension to stack volumes along.
# Choose which volume to rebuild; to_datatree() then NaN-fills the gates that
# were filtered out, restoring the complete azimuth x range grid.
volumes = rdf.data["volume_time"].unique().sort().to_list()
print(f"{len(volumes)} volumes loaded -> rebuilding the first one\n")

dt = rdf.to_datatree(timestep=volumes[0])
dt

30 volumes loaded -> rebuilding the first one



<xarray.DataTree>
Group: /
├── Group: /sweep_0
│       Dimensions:          (azimuth: 360, range: 500)
│       Coordinates: (12/15)
│         * azimuth          (azimuth) float64 3kB 0.1 1.1 2.1 3.1 ... 357.1 358.1 359.1
│         * range            (range) float32 2kB 250.0 750.0 ... 2.492e+05 2.498e+05
│           latitude         (azimuth, range) float64 1MB 60.91 60.91 ... 63.14 63.15
│           longitude        (azimuth, range) float64 1MB 27.11 27.11 ... 27.04 27.04
│           altitude         (azimuth, range) float64 1MB 140.3 143.0 ... 5.117e+03
│           x                (azimuth, range) float64 1MB 0.4363 1.309 ... -3.921e+03
│           ...               ...
│           y_3067           (azimuth, range) float64 1MB 6.752e+06 ... 7.002e+06
│           site_latitude    float64 8B 60.9
│           site_longitude   float64 8B 27.11
│           site_altitude    float64 8B 139.0
│           sweep_number     int64 8B 0
│           elevation_angle  float64 8B 0.3
│       Data variables: (12/20)
│           time             (azimuth, range) datetime64[ns] 1MB NaT NaT NaT ... NaT NaT
│           CSP              (azimuth, range) float32 720kB nan nan nan ... nan nan nan
│           DBZH             (azimuth, range) float32 720kB nan nan nan ... nan nan nan
│           DBZHC            (azimuth, range) float32 720kB nan nan nan ... nan nan nan
│           DBZV             (azimuth, range) float32 720kB nan nan nan ... nan nan nan
│           HCLASS           (azimuth, range) float32 720kB nan nan nan ... nan nan nan
│           ...               ...
│           TV               (azimuth, range) float32 720kB nan nan nan ... nan nan nan
│           VRADDH           (azimuth, range) float32 720kB nan nan nan ... nan nan nan
│           VRADH            (azimuth, range) float32 720kB nan nan nan ... nan nan nan
│           WRADH            (azimuth, range) float32 720kB nan nan nan ... nan nan nan
│           ZDR              (azimuth, range) float32 720kB nan nan nan ... nan nan nan
│           ZDRC             (azimuth, range) float32 720kB nan nan nan ... nan nan nan
├── Group: /sweep_1
│       Dimensions:          (azimuth: 360, range: 500)
│       Coordinates: (12/15)
│         * azimuth          (azimuth) float64 3kB 0.0 1.0 2.0 3.0 ... 357.0 358.0 359.0
│         * range            (range) float32 2kB 250.0 750.0 ... 2.492e+05 2.498e+05
│           latitude         (azimuth, range) float64 1MB 60.91 60.91 ... 63.14 63.15
│           longitude        (azimuth, range) float64 1MB 27.11 27.11 ... 27.03 27.03
│           altitude         (azimuth, range) float64 1MB 142.1 148.2 ... 6.859e+03
│           x                (azimuth, range) float64 1MB 0.0 0.0 ... -4.356e+03
│           ...               ...
│           y_3067           (azimuth, range) float64 1MB 6.752e+06 ... 7.002e+06
│           site_latitude    float64 8B 60.9
│           site_longitude   float64 8B 27.11
│           site_altitude    float64 8B 139.0
│           sweep_number     int64 8B 1
│           elevation_angle  float64 8B 0.7
│       Data variables: (12/20)
│           time             (azimuth, range) datetime64[ns] 1MB 2024-06-10T12:00:29....
│           CSP              (azimuth, range) float32 720kB 15.3 nan nan ... nan nan nan
│           DBZH             (azimuth, range) float32 720kB 0.54 nan nan ... nan nan nan
│           DBZHC            (azimuth, range) float32 720kB 0.54 nan nan ... nan nan nan
│           DBZV             (azimuth, range) float32 720kB 15.18 nan nan ... nan nan
│           HCLASS           (azimuth, range) float32 720kB 1.0 nan nan ... nan nan nan
│           ...               ...
│           TV               (azimuth, range) float32 720kB -327.7 nan nan ... nan nan
│           VRADDH           (azimuth, range) float32 720kB 3.39 nan nan ... nan nan nan
│           VRADH            (azimuth, range) float32 720kB 0.69 nan nan ... nan nan nan
│           WRADH            (azimuth, range) float32 720kB 0.01 nan nan ... n

---
**Next:** [3 — Areas of interest](03_area_of_interest.ipynb)